In [1]:
module BlockSparseArrays

using ArgCheck
using Base: oneto
using LinearAlgebra
using SparseArrays

export BlockSparseMatrixCSC, blocksparse

function half(i::I) where {I}
    return div(i, convert(I, 2))
end

function ispositive(i::I) where {I}
    return i > zero(I)
end

struct BlockSparseMatrixCSC{T, I} <: AbstractMatrix{T}
    nrow::I
    ncol::I
    nadj::I
    nrowblk::I
    ncolblk::I
    nadjblk::I
    xadj::Vector{I}
    xrowblk::Vector{I}
    xcolblk::Vector{I}
    xadjblk::Vector{I}
    adj::Vector{I}
    adjblk::Vector{T}

    function BlockSparseMatrixCSC{T, I}(
            nrow::Integer, 
            ncol::Integer,
            nadj::Integer,
            nrowblk::Integer,
            ncolblk::Integer,
            nadjblk::Integer,
            xadj::AbstractVector,
            xrowblk::AbstractVector,
            xcolblk::AbstractVector,
            xadjblk::AbstractVector,
            adj::AbstractVector,
            adjblk::AbstractVector,
        ) where {T, I}
        @argcheck 0 <= nrow
        @argcheck 0 <= ncol
        @argcheck 0 <= nadj
        @argcheck 0 <= nrowblk
        @argcheck 0 <= ncolblk
        @argcheck 0 <= nadjblk
        @argcheck ncol < length(xadj)
        @argcheck ncol < length(xadjblk)
        @argcheck nadj <= length(adj)
        @argcheck nadj <= length(adjblk)
        
        return new{T, I}(nrow, ncol, nadj, nrowblk, ncolblk,
            nadjblk, xadj, xrowblk, xcolblk, xadjblk, adj, adjblk)
    end
end

function SparseArrays.sparse(A::BlockSparseMatrixCSC{T, I}) where {T, I}
    m = A.nrowblk
    n = A.ncolblk
    nnz = A.nadjblk
    
    colptr = Vector{I}(undef, n + one(I))
    rowval = Vector{I}(undef, nnz)
    nzval = Vector{T}(undef, nnz)

    colptr[one(I)] = pblk = one(I)

    for j in oneto(A.ncol)
        colblkstrt = A.xcolblk[j]
        colblkstop = A.xcolblk[j + one(I)] - one(I)
        
        adjstrt = A.xadj[j]
        adjstop = A.xadj[j + one(I)] - one(I)

        for jblk in colblkstrt:colblkstop
            for p in adjstrt:adjstop
                i = A.adj[p]
    
                rowblkstrt = A.xrowblk[i]
                rowblkstop = A.xrowblk[i + one(I)] - one(I)
                    
                adjblkstrt = A.xadjblk[p]
                adjblkstop = A.xadjblk[p + one(I)] - one(I)

                Ap = reshape(
                    view(A.adjblk, adjblkstrt:adjblkstop),
                    rowblkstop - rowblkstrt + one(I),
                    colblkstop - colblkstrt + one(I),
                )
                
                for iblk in rowblkstrt:rowblkstop
                    ip = iblk - rowblkstrt + one(I)
                    jp = jblk - colblkstrt + one(I)

                    rowval[pblk] = iblk
                    nzval[pblk] = Ap[ip, jp]
                    pblk += one(I)
                end
            end

            colptr[jblk + one(I)] = pblk
        end
    end

    return SparseMatrixCSC{T, I}(m, n, colptr, rowval, nzval)
end

function blocksparse(I, J, V, args...)
    matrix = sparse(I, J, eachindex(V), args...)
    return blocksparse(matrix, V)
end

function blocksparse(matrix::SparseMatrixCSC{<:Any, I}, V::AbstractVector{<:AbstractMatrix{T}}) where {T, I}
    nrow = convert(I, size(matrix, 1))
    ncol = convert(I, size(matrix, 2))
    nadj = convert(I, nnz(matrix))

    nrowblk = zero(I)
    ncolblk = zero(I)
    nadjblk = zero(I)
    
    xrowblk = Vector{I}(undef, nrow + one(I))
    xcolblk = Vector{I}(undef, ncol + one(I))
    xadjblk = Vector{I}(undef, nadj + one(I))

    xrowblk[one(I)] = nrowblk + one(I)
    xcolblk[one(I)] = ncolblk + one(I)
    xadjblk[one(I)] = nadjblk + one(I)

    for i in oneto(nrow)
        xrowblk[i + one(I)] = zero(I)
    end

    for i in oneto(ncol)
        xcolblk[i + one(I)] = zero(I)
    end
    
    xadj = matrix.colptr
    adj = matrix.rowval
    val = matrix.nzval

    for p in oneto(nadj)
        A = V[val[p]]
        nadjblk += convert(I, size(A, 1) * size(A, 2))
        xadjblk[p + one(I)] = nadjblk + one(I)
    end

    adjblk = Vector{T}(undef, nadjblk)
    
    for j in oneto(ncol)
        adjstrt = xadj[j]
        adjstop = xadj[j + one(I)] - one(I)

        for p in adjstrt:adjstop
            i = adj[p]; A = V[val[p]]

            xrowblk[i + one(I)] = rowblkdeg = convert(I, size(A, 1))
            xcolblk[j + one(I)] = colblkdeg = convert(I, size(A, 2))

            adjblkstrt = xadjblk[p]
            adjblkstop = xadjblk[p + one(I)] - one(I)

            B = reshape(
                view(adjblk, adjblkstrt:adjblkstop),
                rowblkdeg,
                colblkdeg,
            )

            copyto!(B, A)
        end
    end

    for i in oneto(nrow)
        nrowblk += xrowblk[i + one(I)]
        xrowblk[i + one(I)] = nrowblk + one(I)
    end

    for i in oneto(ncol)
        ncolblk += xcolblk[i + one(I)]
        xcolblk[i + one(I)] = ncolblk + one(I)
    end

    return BlockSparseMatrixCSC{T, I}(nrow, ncol, nadj, nrowblk,
        ncolblk, nadjblk, xadj, xrowblk, xcolblk, xadjblk, adj, adjblk)
end

function bspgetidx(A::BlockSparseMatrixCSC{T, I}, blki::I, blkj::I) where {T, I}
    @boundscheck checkbounds(axes(A, 1), blki)
    @boundscheck checkbounds(axes(A, 2), blkj)

    Aij = zero(T)

    jstrt = one(I)
    jstop = A.ncol

    while jstrt <= jstop
        jcent = jstrt + half(jstop - jstrt)

        if A.xcolblk[jcent] <= blkj
            jstrt = jcent + one(I)
        else
            jstop = jcent - one(I)
        end
    end

    j = jstrt - one(I)
    
    colblkstrt = A.xcolblk[j]
    colblkstop = A.xcolblk[j + one(I)] - one(I)
    
    adjstrt = A.xadj[j]
    adjstop = A.xadj[j + one(I)] - one(I)

    pstrt = adjstrt
    pstop = adjstop

    while pstrt <= pstop
        pcent = pstrt + half(pstop - pstrt)

        if A.xrowblk[A.adj[pcent]] <= blki
            pstrt = pcent + one(I)
        else
            pstop = pcent - one(I)
        end
    end

    p = pstrt - one(I)
    
    if p >= adjstrt
        i = A.adj[p]
        
        rowblkstrt = A.xrowblk[i]
        rowblkstop = A.xrowblk[i + one(I)] - one(I)
    
        if blki <= rowblkstop
            adjblkstrt = A.xadjblk[p]
            adjblkstop = A.xadjblk[p + one(I)] - one(I)
    
            ip = blki - rowblkstrt + one(I)
            jp = blkj - colblkstrt + one(I)
        
            Ap = reshape(
                view(A.adjblk, adjblkstrt:adjblkstop),
                rowblkstop - rowblkstrt + one(I),
                colblkstop - colblkstrt + one(I),
            )
    
            Aij = Ap[ip, jp]
        end
    end
    
    return Aij
end

function bspvecmul_N!(
        C::AbstractVector,
        A::BlockSparseMatrixCSC{<:Any, I},
        B::AbstractVector,
        α::Number,
        β::Number,
    ) where {I}    
    @argcheck size(A, 1) == length(C)
    @argcheck size(A, 2) == length(B)

    C .*= β

    for j in oneto(A.ncol)
        colblkstrt = A.xcolblk[j]
        colblkstop = A.xcolblk[j + one(I)] - one(I)
        
        Bj = view(B, colblkstrt:colblkstop)

        adjstrt = A.xadj[j]
        adjstop = A.xadj[j + one(I)] - one(I)
        
        for p in adjstrt:adjstop
            i = A.adj[p]

            rowblkstrt = A.xrowblk[i]
            rowblkstop = A.xrowblk[i + one(I)] - one(I)
            
            Ci = view(C, rowblkstrt:rowblkstop)

            adjblkstrt = A.xadjblk[p]
            adjblkstop = A.xadjblk[p + one(I)] - one(I)

            Ap = reshape(
                view(A.adjblk, adjblkstrt:adjblkstop),
                rowblkstop - rowblkstrt + one(I),
                colblkstop - colblkstrt + one(I),
            )

            mul!(Ci, Ap, Bj, α, true)
        end
    end

    return C
end

function bspvecmul_TC!(
        f::Function,
        C::AbstractVector,
        A::BlockSparseMatrixCSC{<:Any, I},
        B::AbstractVector,
        α::Number,
        β::Number,
    ) where {I}
    @argcheck size(A, 2) == length(C)
    @argcheck size(A, 1) == length(B)

    C .*= β

    for j in oneto(A.ncol)
        colblkstrt = A.xcolblk[j]
        colblkstop = A.xcolblk[j + one(I)] - one(I)
        
        Cj = view(C, colblkstrt:colblkstop)

        adjstrt = A.xadj[j]
        adjstop = A.xadj[j + one(I)] - one(I)
        
        for p in adjstrt:adjstop
            i = A.adj[p]

            rowblkstrt = A.xrowblk[i]
            rowblkstop = A.xrowblk[i + one(I)] - one(I)
            
            Bi = view(B, rowblkstrt:rowblkstop)

            adjblkstrt = A.xadjblk[p]
            adjblkstop = A.xadjblk[p + one(I)] - one(I)

            Ap = reshape(
                view(A.adjblk, adjblkstrt:adjblkstop),
                rowblkstop - rowblkstrt + one(I),
                colblkstop - colblkstrt + one(I),
            )

            mul!(Cj, Ap |> f, Bi, α, true)
        end
    end

    return C
end

# ======================== #
# Abstract Array Interface #
# ======================== #

function LinearAlgebra.mul!(
        C::AbstractVector,
        A::BlockSparseMatrixCSC,
        B::AbstractVector,
        α::Number,
        β::Number,
    )
    bspvecmul_N!(C, A, B, α, β)
    return C
end

function LinearAlgebra.mul!(
        C::AbstractVector,
        A::Transpose{<:Any, <:BlockSparseMatrixCSC},
        B::AbstractVector,
        α::Number,
        β::Number,
    )
    bspvecmul_TC!(transpose, C, parent(A), B, α, β)
    return C
end

function LinearAlgebra.mul!(
        C::AbstractVector,
        A::Adjoint{<:Any, <:BlockSparseMatrixCSC},
        B::AbstractVector,
        α::Number,
        β::Number,
    )
    bspvecmul_TC!(adjoint, C, parent(A), B, α, β)
    return C
end

function Base.getindex(A::BlockSparseMatrixCSC{<:Any, I}, i::Integer, j::Integer) where {I}
    return bspgetidx(A, convert(I, i), convert(I, j))
end

function Base.IndexStyle(::Type{<:BlockSparseMatrixCSC})
    return IndexCartesian()
end

function Base.size(A::BlockSparseMatrixCSC)
    m = convert(Int, A.nrowblk)
    n = convert(Int, A.ncolblk)
    return (m, n)
end

end

Main.BlockSparseArrays

In [25]:
using .BlockSparseArrays
using AlgebraicOptimization
using MatrixMarket
using SparseArrays
using Graphs
using SuiteSparseMatrixCollection

# construct the database
# http://sparse.tamu.edu
ssmc = ssmc_db()

# the name of the graph to fetch
#name = "fe_tooth"
name = "bcsstk01"

# fetch the graph
graph = mmread(joinpath(fetch_ssmc(ssmc[ssmc.name.==name, :], format="MM")[1], "$(name).mtx"))

# remove self edges
fkeep!((i, j, v) -> i != j, graph)

# remove weights
fill!(nonzeros(graph), 1)

println(repr("text/plain", graph))

g = Graph(graph)

48×48 SparseMatrixCSC{Float64, Int64} with 352 stored entries:
⎡⢀⡰⠝⢑⢔⠁⠀⠀⠀⠑⢄⠔⠑⣤⠈⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⢗⢁⠄⠁⢀⢵⢄⠀⢀⠐⠁⠑⠂⠀⠑⢄⠀⠠⠀⠀⠀⠀⠀⠀⎥
⎢⠔⠑⢄⣔⠎⠁⢀⠕⢅⠀⠀⠀⠀⠀⠀⡀⠛⢄⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠑⢄⠔⢠⡲⠝⢑⢔⠁⠀⠀⠀⠀⠀⠀⠑⣤⠈⢱⣶⠉⎥
⎢⢄⠀⢀⠐⠁⠑⢗⢁⠎⠁⢀⢵⠀⠀⠀⠀⠀⠀⠂⠀⠑⢇⠀⠻⎥
⎢⢀⠕⢅⠀⠀⠀⠔⠑⢄⣔⠎⠁⠀⠀⠀⠀⠀⠀⠀⠀⠀⡀⠛⢄⎥
⎢⠑⣤⠈⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⡰⠉⢑⢔⠁⠀⠀⠀⠀⠀⠀⎥
⎢⠂⠀⠑⢄⠀⠠⠀⠀⠀⠀⠀⠀⢇⢀⠄⠁⢀⢵⢄⠀⢀⠀⠀⠀⎥
⎢⠀⠀⠀⡀⠛⢄⠀⠀⠀⠀⠀⠀⠔⠑⢄⣔⠎⠁⢀⠕⢅⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠑⣤⠈⠀⠀⠀⠀⠀⠀⠑⢄⠔⢀⡰⠝⢑⢔⠁⎥
⎢⠀⠀⠀⠀⠀⠀⢆⣀⠵⢄⠀⠠⠀⠀⠀⠐⠁⠑⢗⢁⠄⠁⣀⠵⎥
⎣⠀⠀⠀⠀⠀⠀⡜⠛⣤⡀⠛⢄⠀⠀⠀⠀⠀⠀⠔⠑⢄⡜⠊⡠⎦


{48, 176} undirected simple Int64 graph

In [26]:
node_dims = 50
edge_dims = 20

c = CellularSheaf(repeat([node_dims], nv(g)), repeat([edge_dims], ne(g)))
I = Int64[]
J = Int64[]
V = Matrix{Float64}[]

for (e_idx, e) in zip(1:ne(g), edges(g))
    i = src(e)
    j = dst(e)
    rm1 = rand(edge_dims, node_dims)
    rm2 = rand(edge_dims, node_dims)
    
    push!(J, i, j)
    push!(I, e_idx, e_idx)
    push!(V, rm1, -rm2)
    
    set_edge_maps!(c, i, j, e_idx, rm1, rm2)
    
    
    
end

B = blocksparse(I,J,V)

3520×2400 BlockSparseMatrixCSC{Float64, Int64}:
 0.35438   0.753479   0.0501507  0.695731   …   0.0          0.0
 0.378384  0.323536   0.288289   0.912113       0.0          0.0
 0.696942  0.958292   0.924871   0.494326       0.0          0.0
 0.228209  0.468395   0.992992   0.225531       0.0          0.0
 0.828924  0.192074   0.609914   0.357014       0.0          0.0
 0.23518   0.967181   0.531116   0.0292942  …   0.0          0.0
 0.178282  0.0891962  0.73928    0.748948       0.0          0.0
 0.875428  0.563756   0.158559   0.24615        0.0          0.0
 0.959338  0.164986   0.998254   0.751457       0.0          0.0
 0.316968  0.99338    0.495556   0.761387       0.0          0.0
 0.637617  0.744925   0.639457   0.58378    …   0.0          0.0
 0.945774  0.0246013  0.965664   0.979454       0.0          0.0
 0.185752  0.180168   0.15728    0.75399        0.0          0.0
 ⋮                                          ⋱               
 0.0       0.0        0.0        0.0          

In [29]:
# Test diagonal dominance
C = sparse(B)

L = C'*C

function is_dd(C)
    res = true
    for i in 1:size(C)[1]
        row_sum = 0
        for j in 1:size(C)[2]
            if i != j
                row_sum -= abs(C[i,j])
            else
                row_sum += abs(C[i,j])
            end
        end
        if row_sum < 0
            res = false
            break
        end
    end
    return res
end     
        
is_dd(L)


false

In [30]:
L

2400×2400 SparseMatrixCSC{Float64, Int64} with 1000000 stored entries:
⎡⠿⣧⣠⣼⣿⡿⣤⣠⡿⠇⠀⠀⠀⠀⠀⢿⣤⣀⣀⣤⢿⣤⣀⡀⠿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⣀⣾⣿⣿⡃⣀⣾⢻⣆⡀⠀⠀⠀⠀⠀⠀⢘⣻⣟⡃⠀⢸⣿⣇⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⡿⠉⢨⡿⣯⠀⠀⣭⣿⣧⠀⠀⠀⠀⠸⠿⠉⠉⠿⠿⠀⠀⠉⢿⣤⠀⠀⠀⣤⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⣻⣾⣛⠀⠀⣻⣾⣷⡟⠛⢻⣶⣶⡟⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⢻⣶⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠿⠏⠈⠹⣧⣿⣽⠿⠿⣧⣤⣼⠿⠿⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⢠⣤⠈⠉⠿⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠉⠛⣿⣀⣀⣿⣿⣿⣶⣟⣿⣿⣶⣰⡟⠃⠀⠀⠀⠀⠈⠉⠀⠀⠀⠙⢻⣶⣀⡀⠛⢻⣶⣶⡟⠛⎥
⎢⠀⠀⠀⠀⠀⠀⢸⣿⣿⡇⣼⢿⣿⣿⣧⣴⠾⠻⣦⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠸⠿⣧⢠⣼⠿⠿⣧⣤⎥
⎢⣤⣄⠀⠀⣀⡀⠿⠉⠉⠿⣿⣿⢉⣿⡿⣯⣀⢀⣭⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠸⠿⠀⠀⠉⢿⣿⡀⠀⢿⣿⎥
⎢⠀⢻⣶⣰⡟⠃⠀⠀⠀⠀⢘⣻⣾⡃⠀⢘⣻⣾⣷⡟⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⣷⣶⡆⠀⎥
⎢⠀⣼⠿⠹⣧⡄⠀⠀⠀⠀⠿⠉⠈⠿⣧⣿⣽⠿⠿⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢠⣤⠉⠉⢿⣤⎥
⎢⠛⣷⣀⣀⠛⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢻⣶⣶⡟⠛⢻⣶⣶⡟⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠸⠿⢿⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣼⠿⠿⣧⣤⣼⠿⠿⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠛⠃⠀⠈⠛⣷⣀⠀⠀⣶⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣀⣀⣿⣿⣿⣀⣀⣿⣿⣶⣀⠀⠀⣀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⢻⣶⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⣿⡇⠀⢸⣿⣿⡷⠟⠘⠻⣦⡶⠛⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⣤⠈⠉⠿⣧⣄⠀⠀⠀⣀⡀⠀⠀⠀⠀⠿⠉⠉⠿⣿⣿⣽⠏⠿⣧⣠⣼⠏⣿⣤⣀⠀⠀⣀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢻⣶⣀⡀⠛⠃⠀⠀⠀⠀⠀⠀⠀⠀⠘⢻⣶⡀⣀⣾⢻⣶⣶⣟⣿⢻⣆⣶⡟⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠸⠿⣧⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢨⡿⣯⣥⣼⢿⡿⣯⢠⣼⠏⠿⣧⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⣀⣀⣶⣿⣷⣀⠀⠀⣶⠀⠀⠀⠀⠀⠘⠛⠀⠀⢻⣿⣛⣀⣶⢻⣶⣀⣀⣾⣿⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣿⣿⡇⠀⠈⢹⣿⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢨⣽⣯⡅⠀⢸⣿⣿⡇⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⣿⠉⠉⣿⣿⣷⠈⠉⠛⣷⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⠛⠉⠉⠛⣾⣿⠉⠉⣿⣿⎦

In [31]:
L[1,:]

2400-element SparseVector{Float64, Int64} with 400 stored entries:
  [1   ]  =  49.2503
  [2   ]  =  36.1195
  [3   ]  =  38.5623
  [4   ]  =  36.75
  [5   ]  =  35.1063
  [6   ]  =  41.2644
  [7   ]  =  36.5901
  [8   ]  =  36.2197
  [9   ]  =  38.4301
  [10  ]  =  39.6205
          ⋮
  [1490]  =  -4.74975
  [1491]  =  -6.86342
  [1492]  =  -4.93769
  [1493]  =  -5.9666
  [1494]  =  -6.33066
  [1495]  =  -5.28644
  [1496]  =  -6.50123
  [1497]  =  -6.66654
  [1498]  =  -5.34777
  [1499]  =  -5.83672
  [1500]  =  -5.49127

In [19]:
x = rand(size(B, 2))

42000-element Vector{Float64}:
 0.793883649103356
 0.42576183818376623
 0.605917755783115
 0.022909755496615003
 0.9405501847969983
 0.27262259620849394
 0.05713232709631744
 0.6134081534073166
 0.14479191324173601
 0.7999896395913718
 0.11178148652280162
 0.28141878673842824
 0.5271983163130005
 ⋮
 0.16552802720752113
 0.8551554491742659
 0.6218791499482124
 0.10798076925088418
 0.6771016010021185
 0.25237345305019887
 0.7377671786206644
 0.4557822173809466
 0.741788127171375
 0.8629026911145662
 0.876536512818602
 0.08939913491947427

In [21]:
@time B'*(B*x);

  0.022279 seconds (37.21 k allocations: 3.670 MiB)


In [7]:
A = c.coboundary;

In [12]:
@time A'*(A*x);

  7.602598 seconds (11 allocations: 2.016 MiB)


In [10]:
using LinearAlgebra

norm(A'*(A*x)-B'*(B*x))

8.709216461679173e-12

In [22]:
Bsp = sparse(B)

186000×42000 SparseMatrixCSC{Float64, Int64} with 37200000 stored entries:
⎡⣿⢿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⢸⣼⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠘⡏⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⢻⢹⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⢸⡿⡀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠈⡇⡇⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⢹⢹⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠸⡿⡇⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⣧⣧⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⢹⣽⡄⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠈⣧⣇⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⢻⣽⡀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠘⣧⡇⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⢻⣻⡀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠘⣟⡇⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⢻⣻⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠘⣇⡇⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⢻⢱⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠘⡏⡆⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⣷⢷⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⢸⣸⡀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠈⣏⣇⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⢿⢻⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⣾⡄⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⣷⠃⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠸⡆⎦

In [24]:
@time Bsp'*(Bsp*x)

  0.062933 seconds (7 allocations: 1.740 MiB)


42000-element Vector{Float64}:
 -66.86836518124275
 -66.7381359255755
 -62.10956405185443
 -70.56113031159202
 -61.59591440447521
 -76.58618350976775
 -75.66324505900138
 -66.72736521423369
 -79.71384070982134
 -48.85334378187375
 -83.61382481052401
 -71.34528492696519
 -74.42118207852272
   ⋮
 130.1290780947282
 160.28517671303672
 133.92133251031504
 138.3634498842349
 134.23170088973316
 115.63493872569023
 108.60360159298399
 118.39962940436118
 117.5166216716489
 136.97163031051764
 127.25559629758872
 118.27519034612983